> **Experiment 1 — Research agent.** Repeated SMILES → PubChem → ECHA CHEM resolution for a fixed test molecule (Toluene).

# SMILES → ECHA Hazard Check with Three Agents

A minimal workshop version of the working V2 workflow:

1. **Research & Browser Agent** — SMILES → PubChem → ECHA, then cookie/legal-notice preparation.
2. **PixelRAG Agent** — uses the exact V2 `render_url(..., cdp_url=...)` capture path.
3. **Safety Agent** — uses the V2 vision-risk schema to read visible ECHA classifications.

The PixelRAG installation is pinned to the exact commit recorded in the successful V2 execution.


## 1. Install PixelRAG

In [ ]:
# Kaggle Internet must be enabled.
# Reproduce the exact PixelRAG revision recorded in the successful V2 run.
!python -m pip uninstall -y playwright >/dev/null 2>&1 || true

!rm -rf /kaggle/working/PixelRAG
!git clone https://github.com/StarTrail-org/PixelRAG.git /kaggle/working/PixelRAG
!git -C /kaggle/working/PixelRAG checkout --detach 7a3c3041bd37b1868d8573039ebdc60607ddb936

%pip -q install -U \
    pillow \
    websockets \
    requests \
    openai \
    langgraph \
    langchain-core \
    typing_extensions \
    zstandard

%pip -q install -e /kaggle/working/PixelRAG --no-deps

!PYTHONPATH=/kaggle/working/PixelRAG/render/src \
 python -m pixelrag_render.chrome install

!PYTHONPATH=/kaggle/working/PixelRAG/render/src \
 python -m pixelrag_render.chrome which

PIXELRAG_COMMIT = !git -C /kaggle/working/PixelRAG rev-parse HEAD
PIXELRAG_COMMIT = PIXELRAG_COMMIT[0].strip()
print("PixelRAG commit:", PIXELRAG_COMMIT)
assert PIXELRAG_COMMIT == "7a3c3041bd37b1868d8573039ebdc60607ddb936"
print("Pinned V2 PixelRAG installation completed")


## 2. Load the Utility Module

In [ ]:
import importlib, sys
from pathlib import Path

# Looks for the utility module in the repo (../src) first, then in a Kaggle utility-script mount.
CANDIDATES = [
    Path.cwd().parent / "src",
    Path.cwd() / "src",
    Path("/kaggle/usr/lib/notebooks/burkaytuntrk/agentic_smiles_hazard_utils"),
]
UTILS_DIR = next((p for p in CANDIDATES if (p / "agentic_smiles_hazard_utils.py").exists()), None)
if UTILS_DIR is None:
    raise FileNotFoundError("agentic_smiles_hazard_utils.py not found; see README (Setup).")
if str(UTILS_DIR) not in sys.path:
    sys.path.insert(0, str(UTILS_DIR))

sys.modules.pop("agentic_smiles_hazard_utils", None)
utils = importlib.import_module("agentic_smiles_hazard_utils")


## 3. PixelRAG Browser Self-Test

In [ ]:
from agentic_smiles_hazard_utils import pixelrag_self_test

pixelrag_self_test()


## 5. Shared State

In [ ]:
import typing

class AgentState(typing.TypedDict, total=False):
    smiles: str
    identity: dict[str, typing.Any]
    echa_resolution: dict[str, typing.Any]
    browser: dict[str, typing.Any]
    browser_status: str
    capture: dict[str, typing.Any]
    capture_status: str
    capture_attempts: int
    risk: dict[str, typing.Any]
    final_record: dict[str, typing.Any]
    text_report: str
    workflow_succeeded: bool
    error: str


## 6. Agent 1 — Research & Browser Preparation Agent

This agent resolves the SMILES and exact ECHA URL, then prepares the shared browser profile. It dismisses the cookie banner when present, accepts the ECHA legal notice, and verifies that `Industry classifications` / `CLP Submission` are rendered before PixelRAG runs.


In [ ]:
def research_agent(state: AgentState) -> AgentState:
    from agentic_smiles_hazard_utils import (
        browser_session_agent,
        lookup_echa,
        lookup_identity,
    )

    identity = lookup_identity(state["smiles"])
    if identity.get("status") != "resolved":
        return {
            "identity": identity,
            "browser_status": "failed",
            "workflow_succeeded": False,
            "error": identity.get("error", "Identity lookup failed."),
        }

    resolution = lookup_echa(identity)
    if resolution.get("status") != "found":
        return {
            "identity": identity,
            "echa_resolution": resolution,
            "browser_status": "failed",
            "workflow_succeeded": False,
            "error": resolution.get("notes", "ECHA record not found."),
        }

    browser_update = browser_session_agent({"echa_resolution": resolution})

    return {
        "identity": identity,
        "echa_resolution": resolution,
        "browser": browser_update.get("browser", {}),
        "browser_status": browser_update.get("browser_status", "failed"),
    }


## 7. Agent 2 — PixelRAG Agent

This deliberately calls the **original V2 capture function unchanged**. It uses `PixelRAG.render_url()` with the prepared Chrome CDP browser—the same path that produced the working Clozapine hazard screenshot in V2.


In [ ]:
def pixelrag_agent(state: AgentState) -> AgentState:
    from agentic_smiles_hazard_utils import pixelrag_capture_agent

    update = pixelrag_capture_agent(state)

    return {
        "capture": update.get("capture", {}),
        "capture_status": update.get("capture_status", "failed"),
        "capture_attempts": update.get(
            "capture_attempts",
            state.get("capture_attempts", 0),
        ),
    }


## 8. Agent 3 — Safety Agent

The hazard-reading part below is copied from the working V2 **Vision Risk Agent**: the same structured schema, evidence rules and image-reading prompt are retained. This is the important part that previously extracted three visible Clozapine classifications from the PixelRAG tile.


In [ ]:
RISK_SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "properties": {
        "evidence_status": {
            "type": "string",
            "enum": ["sufficient", "partial", "insufficient"],
        },
        "substance_match": {
            "type": "string",
            "enum": ["exact", "probable", "not_verified"],
        },
        "substance_name": {"type": "string"},
        "cas": {"type": "string"},
        "ec_number": {"type": "string"},
        "submission": {"type": "string"},
        "notification_share": {"type": "string"},
        "status": {"type": "string"},
        "last_updated": {"type": "string"},
        "signal_word": {"type": "string"},
        "classifications": {
            "type": "array",
            "items": {
                "type": "object",
                "additionalProperties": False,
                "properties": {
                    "hazard_class_category": {"type": "string"},
                    "hazard_statement_code": {"type": "string"},
                    "hazard_statement_text": {"type": "string"},
                    "reason_for_classification": {"type": "string"},
                    "affected_organs_routes": {"type": "string"},
                },
                "required": [
                    "hazard_class_category",
                    "hazard_statement_code",
                    "hazard_statement_text",
                    "reason_for_classification",
                    "affected_organs_routes",
                ],
            },
        },
        "pictograms": {
            "type": "array",
            "items": {"type": "string"},
        },
        "key_risks": {
            "type": "array",
            "items": {"type": "string"},
        },
        "visual_evidence_notes": {"type": "string"},
    },
    "required": [
        "evidence_status",
        "substance_match",
        "substance_name",
        "cas",
        "ec_number",
        "submission",
        "notification_share",
        "status",
        "last_updated",
        "signal_word",
        "classifications",
        "pictograms",
        "key_risks",
        "visual_evidence_notes",
    ],
}

def safety_agent(state: AgentState) -> AgentState:
    # Hazard extraction below is the V2 Vision Risk Agent logic.
    import json
    from pathlib import Path
    from agentic_smiles_hazard_utils import encode_image, structured_response

    identity = state["identity"]
    capture = state["capture"]

    tile_paths = [Path(value) for value in capture.get("tile_paths", [])]
    if not tile_paths:
        return {
            "workflow_succeeded": False,
            "error": "The PixelRAG capture contained no usable tiles.",
        }

    content = [
        {
            "type": "input_text",
            "text": json.dumps(
                {
                    "task": (
                        "Read the supplied PixelRAG screenshots of an ECHA CHEM page "
                        "and extract only risk information visibly supported by them."
                    ),
                    "expected_identity": {
                        "name": identity.get("name", ""),
                        "cas": identity.get("cas", ""),
                        "ec_number": state.get("echa_resolution", {}).get(
                            "matched_ec_number", ""
                        ),
                    },
                    "rules": [
                        "Do not infer hidden fields.",
                        "Do not use general chemical knowledge.",
                        "Mark evidence partial or insufficient when the relevant panel is absent.",
                        "Require visible identity evidence before declaring an exact match.",
                        "Transcribe percentages, dates, codes, labels, and pictograms carefully.",
                    ],
                },
                ensure_ascii=False,
            ),
        }
    ]

    for path in tile_paths:
        content.append(
            {
                "type": "input_image",
                "image_url": encode_image(path),
                "detail": "high",
            }
        )

    risk, _ = structured_response(
        instructions=(
            "You are the Vision Risk Agent. You read PixelRAG screenshot tiles "
            "captured from ECHA CHEM. Extract only visually supported content. "
            "Be conservative and exact."
        ),
        content=content,
        schema_name="pixelrag_echa_risk",
        schema=RISK_SCHEMA,
        use_web_search=False,
    )

    resolution = state.get("echa_resolution", {})
    record = {
        "smiles": state["smiles"],
        "identity": identity,
        "echa_url": resolution.get("industry_url") or resolution.get("substance_url", ""),
        "pixelrag_tiles": [str(path) for path in tile_paths],
        "risk": risk,
    }

    lines = [
        f"Substance: {risk.get('substance_name') or identity.get('name', '')}",
        f"CAS: {risk.get('cas') or identity.get('cas', '')}",
        f"Signal word: {risk.get('signal_word', '')}",
        "",
        "ECHA classifications:",
    ]

    for row in risk.get("classifications", []):
        parts = [
            row.get("hazard_class_category", ""),
            row.get("hazard_statement_code", ""),
            row.get("hazard_statement_text", ""),
        ]
        lines.append("- " + " | ".join(part for part in parts if part))

    if risk.get("pictograms"):
        lines.extend(["", "Pictograms: " + ", ".join(risk["pictograms"])])

    return {
        "risk": risk,
        "final_record": record,
        "text_report": "\n".join(lines),
        "workflow_succeeded": risk.get("evidence_status") != "insufficient",
        "error": (
            ""
            if risk.get("evidence_status") != "insufficient"
            else "The screenshots did not contain sufficient visible ECHA hazard evidence."
        ),
    }
#############################################################################

In [ ]:
#Test Running Cell  —  RESEARCH AGENT ONLY
# PixelRAG and the Safety agent are NOT touched here. No Chrome, no vision call.
import json, re, time
from agentic_smiles_hazard_utils import lookup_identity, lookup_echa, MODEL

# ---------------------------------------------------------------- test case
SMILES            = "Cc1ccccc1"     # Toluene
EXPECTED_CAS      = "108-88-3"
EXPECTED_EC       = "203-625-9"
EXPECTED_URL_PART = "100.003.297"  # e.g. "100.003.297"

print(f"model  : {MODEL}")
print(f"SMILES : {SMILES}")
print("-" * 72)

for x in range(10):
    def norm_cas(v):
        return re.sub(r"[^0-9-]", "", str(v or "")).strip("-")
    
    # ---------------------------------------------------------------- step 1
    t0 = time.time()
    identity = lookup_identity(SMILES) or {}
    t_identity = round(time.time() - t0, 1)
    
    print(f"[1] lookup_identity   ({t_identity}s)   status = {identity.get('status')}")
    print(f"    name : {identity.get('name', '')}")
    print(f"    CAS  : {identity.get('cas', '')}   candidates: {identity.get('cas_candidates', [])[:5]}")
    
    if identity.get("status") != "resolved":
        raise SystemExit(f"identity could not be resolved: {identity.get('error', '')}")
    
    # ---------------------------------------------------------------- step 2
    t0 = time.time()
    resolution = lookup_echa(identity) or {}
    t_echa = round(time.time() - t0, 1)
    
    url = resolution.get("industry_url") or resolution.get("substance_url") or ""
    
    print("-" * 72)
    print(f"[2] lookup_echa       ({t_echa}s)   status = {resolution.get('status')}")
    print(f"    matched_name : {resolution.get('matched_name', '')}")
    print(f"    matched_cas  : {resolution.get('matched_cas', '')}")
    print(f"    EC number    : {resolution.get('matched_ec_number', '')}")
    print(f"    notes        : {resolution.get('notes', '')}")
    print()
    print("    URL          :", url or "(none)")
    for extra in resolution.get("source_urls", []):
        if extra != url:
            print("    also found   :", extra)
    
    # ---------------------------------------------------------------- comparison
    print("-" * 72)
    checks = {}
    checks["echa status is 'found'"] = resolution.get("status") == "found"
    checks["CAS matches expected"]   = norm_cas(resolution.get("matched_cas")) == norm_cas(EXPECTED_CAS)
    checks["URL is a chem.echa URL"] = "chem.echa.europa.eu" in url
    if EXPECTED_EC:
        checks["EC matches expected"] = norm_cas(resolution.get("matched_ec_number")) == norm_cas(EXPECTED_EC)
    if EXPECTED_URL_PART:
        checks["URL is the right record"] = EXPECTED_URL_PART in url
    
    # the hardcoded tutorial shortcut - if this fires, the model was not the one that found it
    used_fallback = "fallback" in str(resolution.get("notes", "")).lower()
    
    print("COMPARISON")
    for name, passed in checks.items():
        print(f"  [{'PASS' if passed else 'FAIL'}]  {name}")
    
    success = all(checks.values())
    print()
    print(f"RESULT   : {'SUCCESS' if success else 'FAILURE'}")
    print(f"expected : CAS {EXPECTED_CAS}" + (f" / EC {EXPECTED_EC}" if EXPECTED_EC else ""))
    print(f"got      : CAS {resolution.get('matched_cas', '') or '-'} / EC {resolution.get('matched_ec_number', '') or '-'}")
    print(f"time     : {round(t_identity + t_echa, 1)}s")
    if used_fallback:
        print("WARNING  : the hardcoded TUTORIAL_ECHA_FALLBACKS entry was used - "
              "this result does not measure the model")
    
    research_output = {"smiles": SMILES, "identity": identity, "echa_resolution": resolution}

## 12. Clean Up the Browser

In [ ]:
from agentic_smiles_hazard_utils import cleanup_browser

cleanup_browser(globals().get("state", {}).get("browser", {}))

try:
    proc.terminate(); proc.wait(timeout=10)
    print("Chrome turned off, pid:", proc.pid)
except Exception as e:
    print("Chrome :", type(e).__name__, e)

## 13. HIP LLM